In [2]:
# Import neded packages

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
import numpy as np
import matplotlib.pyplot as plt
import matplotlib


# #%matplotlib widget
# %matplotlib inline


In [3]:
# We move our tensor to the GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using {} device'.format(device))

Using cuda device


In [4]:
# Defined the DGM network


#__________________________ The class that defines the DGM layer ______________________________

class DGM_layer(nn.Module):
    """
        The parametres:
                        d: dimension of the space domain
                        M: number of units in each layer
                  """

    def __init__(self,d,M):
        super(DGM_layer, self).__init__()
        self.Uz = nn.Linear(d, M, bias=False)
        self.Wzbz = nn.Linear(M, M)
        self.Ug = nn.Linear(d, M, bias=False)
        self.Wgbg = nn.Linear(M, M)
        self.Ur = nn.Linear(d, M, bias=False)
        self.Wrbr = nn.Linear(M, M)
        self.Uh = nn.Linear(d, M, bias=False)
        self.Whbh = nn.Linear(M, M)
        self.onesTens = torch.ones(M).to(device)

    def activation(self, x):
        return torch.tanh(x)
        #return torch.sigmoid(x)
        #return x * torch.sigmoid(x)
        #return torch.relu(x)
        #return torch.cos(x)

    def forward(self, xt, prevS):
        Z = self.activation(self.Uz(xt) + self.Wzbz(prevS))
        G = self.activation(self.Ug(xt) + self.Wgbg(prevS))
        R = self.activation(self.Ur(xt) + self.Wrbr(prevS))
        SR = prevS * R
        H = self.activation(self.Uh(xt) + self.Whbh(SR))
        return (self.onesTens - G) * H + Z * prevS


#__________________________ The class that defines the DGM network ______________________________


class DGM_Net(nn.Module):
    """
        The parametres:
                        d: dimension of the space domain
                        M: number of units in each layer
                        L: number of DGM layers
                        X: the vector of  spatial data
                        t: the vector of time data
                  """

    def __init__(self, d, M, L):
        super(DGM_Net, self).__init__()
        self.initial_layer = nn.Linear(d, M)
        self.middle_layers = nn.ModuleList([DGM_layer(d, M) for i in range(L)])
        self.final_layer = nn.Linear(M, 1)

    def activation(self, x):
        return torch.tanh(x)
        #return torch.sigmoid(x)
        #return x * torch.sigmoid(x)
        #return torch.relu(x)
        #return torch.cos(x)

    def forward(self, X):
        S = self.activation(self.initial_layer(X))
        for i, DGMlayer in enumerate(self.middle_layers):
            S = DGMlayer(X, S)
        return self.final_layer(S)


In [5]:
#_______________ hyperparameters __________________________________

dim = 1                        # dimension of the space domain
M = 20                           # number of units in each layer
L = 3                            # number of DGM layers
num_eps = 5000                  # number of epochs totale

mini_batch_size = 1024

mini_batch_size_bdry = 1000
x_low = 0.0
x_high = 1.0               # domain dimensions

T = 1.0



# used in calculating boundary loss
ones = torch.ones((mini_batch_size_bdry, 1)).to(device)
zeros = torch.zeros((mini_batch_size_bdry, 1)).to(device)

In [6]:
#___________________ the analytical solution __________________________________________________________

def U_exat(X):
    t = X[:,0].reshape(-1,1)
    x = X[:,1].reshape(-1,1)
    return x/(2*t+1)


In [ ]:
#_______________ Mean errors ____________________________________

def absulat_ME(Uext,Upred):
    return  torch.sqrt(((Uext - Upred)**2).mean())

def relative_E(Uext,Upred):
    return torch.sqrt(((Uext - Upred)**2).mean()/((Uext)**2).mean())*100

def linf_error(Uext, Upred):
    return torch.max(torch.abs(Uext - Upred))


In [8]:
# Define the learning rate schedule based on the number of epochs
def lr_lambda(epoch):
    if epoch <= 5000:
        return 1e-2
    elif 5000 < epoch <= 10000:
        return 5e-3
    elif 10000 < epoch <= 20000:
        return 1e-3
    elif 20000 < epoch <= 30000:
        return 5e-4
    elif 30000 < epoch <= 40000:
        return 1e-4
    elif 40000 < epoch <= 45000:
        return 5e-5
    else:
        return 1e-5

In [20]:
#________________________ Training __________________________________________________________________________

u = DGM_Net(dim+1,M,L).to(device)   #  The network that will approach the solution


x_sampler = torch.distributions.uniform.Uniform(x_low, x_high)    #   to obtain the spatial data
t_sampler = torch.distributions.uniform.Uniform(0, T)    #   to obtain the temporal data

# Create the optimizer and a LambdaLR scheduler with the defined learning rate schedule
optimizer = optim.Adam(u.parameters(),lr = 1)                   # Define the optimizer

scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)  # to adjust learning rate


loss_train = np.zeros(num_eps)             # loss of taining inside the domaine
relative_E_losses = np.zeros(num_eps)              # losses initialization

print('\n Using {} device'.format(device))
for ep in range(num_eps):


    x = x_sampler.sample((mini_batch_size, dim))
    t = t_sampler.sample((mini_batch_size, 1))

    t_bndr = t_sampler.sample((mini_batch_size_bdry, 1)).to(device)
    x_0 = x_sampler.sample((mini_batch_size, dim)).to(device)

    xt = torch.cat((t,x),1).to(device)
    xt.requires_grad_()


    # evaluate forward pass, compute derivatives of network with respect to x
    u_grad_t = torch.autograd.grad(u(xt), xt, grad_outputs=torch.ones_like(u(xt)), create_graph=True)[0]
    u_t = (u_grad_t[:,0]).reshape(-1,1)

    u_grad_x = torch.autograd.grad(u(xt)**2, xt, grad_outputs=torch.ones_like(u(xt)**2), create_graph=True)[0]
    u_x = (u_grad_x[:,1]).reshape(-1,1)

    L1 = torch.mean(torch.pow((u_t + u_x), 2))          # Operator differential


    x_bry_1 = torch.cat((t_bndr,ones),1)
    L2 = torch.mean(torch.pow(u(x_bry_1) - 1/(2*t_bndr+1), 2))
    x_bry_2 = torch.cat((t_bndr,zeros),1).to(device)
    L3 = torch.mean(torch.pow(u(x_bry_2), 2))                     # Boundary conditions

    t_0 = torch.zeros((mini_batch_size, 1)).to(device)
    xt_0 = torch.cat((t_0,x_0),1)
    xt_0 = torch.Tensor(xt_0)


    L4 = torch.mean(torch.pow((u(xt_0) - x_0), 2))          # Initial conditions

    Loss = L1 + L2 + L3 + L4

    relative_E_losses[ep] =  float(relative_E( U_exat(xt) ,u(xt)).item())

    optimizer.zero_grad()
    Loss.backward()            # compute derivative of loss with respect to network parameters
    optimizer.step()           # update network parameters with ADAM

    # Update the learning rate
    scheduler.step(ep)  # Pass the current epoch to the scheduler

    # Display the current learning rate
    current_lr = optimizer.param_groups[0]['lr']

    loss_train[ep] = float(Loss.item())
    if ep % 10 == 9:
        print("Epoch %d - L_r:%f  loss_int : %.2f - REN : %.2f"%(ep, current_lr, loss_train[ep], relative_E_losses[ep]),"%")



 Using cuda device


/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:204: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Epoch 9 - L_r:0.010000  loss_int : 0.05 - REN : 28.02 %
Epoch 19 - L_r:0.010000  loss_int : 0.02 - REN : 11.40 %
Epoch 29 - L_r:0.010000  loss_int : 0.01 - REN : 6.97 %
Epoch 39 - L_r:0.010000  loss_int : 0.00 - REN : 4.13 %
Epoch 49 - L_r:0.010000  loss_int : 0.00 - REN : 2.05 %
Epoch 59 - L_r:0.010000  loss_int : 0.00 - REN : 2.09 %
Epoch 69 - L_r:0.010000  loss_int : 0.00 - REN : 2.56 %
Epoch 79 - L_r:0.010000  loss_int : 0.00 - REN : 1.57 %
Epoch 89 - L_r:0.010000  loss_int : 0.00 - REN : 1.42 %
Epoch 99 - L_r:0.010000  loss_int : 0.00 - REN : 1.43 %
Epoch 109 - L_r:0.010000  loss_int : 0.00 - REN : 1.22 %
Epoch 119 - L_r:0.010000  loss_int : 0.00 - REN : 1.09 %
Epoch 129 - L_r:0.010000  loss_int : 0.00 - REN : 1.11 %
Epoch 139 - L_r:0.010000  loss_int : 0.00 - REN : 0.99 %
Epoch 149 - L_r:0.010000  loss_int : 0.00 - REN : 1.02 %
Epoch 159 - L_r:0.010000  loss_int : 0.00 - REN : 0.93 %
Epoch 169 - L_r:0.010000  loss_int : 0.00 - REN : 0.93 %
Epoch 179 - L_r:0.010000  loss_int : 0.0

In [1]:

#_______________________________________________________________________________

grid_nums = 500
x_test = torch.distributions.uniform.Uniform(x_low, x_high).sample((grid_nums, dim)).to(device)
t_test = torch.distributions.uniform.Uniform(0, T).sample((grid_nums, 1)).to(device)
X_test = torch.cat((t_test,x_test),1)
with torch.no_grad():
    U = u(X_test).detach()

U_ext =  U_exat(X_test)

print("The Mean Absulat Error: {}".format(absulat_ME(U_ext,U)))
print("The Relative Error: {} %".format(relative_E(U_ext,U)))
print("The L_inf Error: {} %".format(linf_error(U_ext,U)))

NameError: name 'torch' is not defined

In [ ]:
# #_______________________ Save Model _____________________________________

# torch.save(u.state_dict(), "Model_DGM_Heat_equation") # to save weights
# np.save("loss_DGM_Heat_equation" ,Losse)                # to save Losse
# np.save("loss_resudPDE_DGM_Heat_equation" ,Losse_resud)                # to save Losse resudel PDE


In [ ]:
# # # #_______________________ load Model _____________________________________

# u_IS = DGM_Net(dim,M,L).to(device)
# state_dict = torch.load( "Model_IS_DGM_Poisson_special_50000_2.pth")
# u_IS.load_state_dict(state_dict)
# losse_IS = np.load("loss_IS_DGM_Poisson_special_50000_2.npy")

In [ ]:
# # #_______________ Ploting results __________________________________


# grid_nums = 50

# x_grid = torch.linspace(-1.0,1.0,grid_nums)
# XX, YY = torch.meshgrid(x_grid,x_grid)
# X = torch.reshape(XX, (-1,1))
# Y = torch.reshape(YY, (-1,1))
# XY = torch.cat((X,Y),1).to(device)
# with torch.no_grad():
#     U = u_IS(XY)

# UU = torch.reshape(U, (grid_nums,grid_nums)).detach().cpu().numpy()
# U_ext =  U_exat(XY)
# U_ext = torch.reshape(U_ext , (grid_nums,grid_nums)).detach().cpu().numpy()

# fig = plt.figure()
# ax = fig.add_subplot(projection='3d')
# ax.plot_surface(XX.numpy(),YY.numpy(), UU )


In [ ]:
# grid_nums = 500
# x_test = torch.distributions.uniform.Uniform(x_low, x_high).sample((grid_nums, dim)).to(device)
# with torch.no_grad():
#     U = u(x_test).detach()
#     U_IS = u_IS(x_test).detach()


# U_ext =  U_exat(x_test)

# print("MAE_standard_DGM = {} | MAE_Proposed method = {}".format(absulat_ME(U_ext,U), absulat_ME(U_ext,U_IS)))

# print("MRE_standard_DGM = {}% | MRE_Proposed method = {}%".format(relative_E(U_ext,U), relative_E(U_ext,U_IS)))

In [ ]:
# Losse_IS  = np.log(losse_IS)
# Losse_standard  = np.log(Losse)

# x_error = np.linspace(0,50000, 50000).reshape(-1,1)
# plt.plot(x_error, Losse_IS,  label='Losse_IS')
# plt.plot(x_error, Losse_standard, label='Losse_standard')
# plt.legend()
# plt.show